# Imports

In [22]:
import re
import torch
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from category_encoders import CatBoostEncoder

from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import TargetEncoder, StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold

from feature_engine.selection import DropFeatures
from pytorch_tabnet.tab_model import TabNetClassifier

## Utils

In [3]:
set_config(enable_metadata_routing=True)

In [4]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

In [10]:
column_transformer = ColumnTransformer([
    (
        'target_encoder', 
        TargetEncoder(), 
        ['driver', 'compound', 'race']
    ),
    (
        'catboost_encoder', 
        CatBoostEncoder(), 
        ['driver', 'compound', 'race']
    ),
    (
        'standard_scaler', 
        StandardScaler(), 
        ['lapnumber', 'position', 'raceprogress', 'year', 'position_norm', 'race_progress_sin', 'position_vs_mean']
    ),
    (
        'robust_scaler', 
        RobustScaler(), 
        [
            'position_change', 'cumulative_degradation', 'laptime_delta', 'laptime_s', 'stint', 'driver_mean_lap', 'tyrelife', 'delta_x_tyre_life', 
            'compound_tyre_life', 'stint_progress', 'tyre_life_ratio', 'degradation_per_lap', 'position_change_cum', 'laps_since_pit', 'lap_time_inv',  
            'lap_time_vs_race_mean', 'lap_time_x_tyre', 'position_x_progress', 'degradation_x_progress', 'race_progress_squared', 'driver_avg_position' 
        ]
    ),
], remainder="passthrough")

# Loading Dataset

In [5]:
X_train = pd.read_parquet('../data/X_train.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

In [6]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


# Machine Learning

In [25]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):

    print(f"\n======== Fold {fold + 1} ========")

    X_tr, X_val = X_train.iloc[train_idx, :], X_train.iloc[valid_idx, :]
    y_tr, y_val = y_train.iloc[train_idx, 0], y_train.iloc[valid_idx, 0]

    X_tr = column_transformer.fit_transform(X_tr, y_tr)
    X_val = column_transformer.transform(X_val)
    
    model = TabNetClassifier(
        n_d=32,
        n_a=32,
        n_steps=5,
        gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size": 20, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type="entmax",
        seed=42,
        verbose=1
    )

    model.fit(
        X_train=X_tr,
        y_train=y_tr,
        eval_set=[(X_val, y_val)],
        eval_name=["valid"],
        eval_metric=["auc"],
        max_epochs=200,
        patience=20,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=6,
        drop_last=False
    )

    score = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, score)

    aucs.append(auc)

    print(f"Fold AUC: {auc:.6f}")
    

print("\n==============================")
print("CV AUC:", np.mean(aucs))
print("STD:", np.std(aucs))
print("==============================")


======== Fold 1 ========


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.3783  | valid_auc: 0.88995 |  0:00:16s
epoch 1  | loss: 0.30844 | valid_auc: 0.91104 |  0:00:33s
epoch 2  | loss: 0.2897  | valid_auc: 0.91608 |  0:00:50s
epoch 3  | loss: 0.28042 | valid_auc: 0.92223 |  0:01:08s
epoch 4  | loss: 0.26954 | valid_auc: 0.92392 |  0:01:25s
epoch 5  | loss: 0.26764 | valid_auc: 0.92707 |  0:01:41s
epoch 6  | loss: 0.26454 | valid_auc: 0.92843 |  0:01:59s
epoch 7  | loss: 0.26256 | valid_auc: 0.93071 |  0:02:17s
epoch 8  | loss: 0.26235 | valid_auc: 0.92732 |  0:02:35s
epoch 9  | loss: 0.26198 | valid_auc: 0.93071 |  0:02:53s
epoch 10 | loss: 0.25881 | valid_auc: 0.93176 |  0:03:10s
epoch 11 | loss: 0.25781 | valid_auc: 0.93371 |  0:03:27s
epoch 12 | loss: 0.25476 | valid_auc: 0.93339 |  0:03:44s
epoch 13 | loss: 0.25483 | valid_auc: 0.93472 |  0:04:00s
epoch 14 | loss: 0.25266 | valid_auc: 0.93637 |  0:04:17s
epoch 15 | loss: 0.25209 | valid_auc: 0.93114 |  0:04:35s
epoch 16 | loss: 0.25054 | valid_auc: 0.93662 |  0:04:53s
epoch 17 | los

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Fold AUC: 0.943317

======== Fold 2 ========


...

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Fold AUC: 0.941657

======== Fold 3 ========


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.36911 | valid_auc: 0.89207 |  0:00:17s


KeyboardInterrupt: 

# Deploy

In [8]:
dump_pickle(voting, '../models/model_voting.pkl')